# Funcionalidades Auxiliares


### Obtención de url para Quirk a partir de un json register

Utilidad de prueba: a partir de un registro de algoritmo definido como variable string (json), genera la URL de Quirk codificada y funcional para "https://algassert.com/quirk", validando antes que el circuito se decodifica correctamente (round-trip).


In [5]:
# Parámetro de entrada para la celda subsiguiente
url_quirk = (
    'https://algassert.com/quirk#circuit={\"cols\": [[\"H\", 1], [\"Z\", 1], [\"\\u2296\", \"X\"], [\"H\", 1], [\"Measure\", \"Measure\"]]}'
)


In [6]:
# ----------------------------------------------------------------------------
# Utilidad de prueba: recibe la URL de Quirk (aquí, la salida percent-encoded
# de la celda anterior) y la limpia/formatea para que el circuito se vea
# correctamente en https://algassert.com/quirk.
# ----------------------------------------------------------------------------

import sys
from pathlib import Path
from IPython.display import HTML, display

# La raíz del proyecto se resuelve desde la ubicación del cuadernillo, igual
# que en QTrans.ipynb, para poder importar el paquete ``qtrans``.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / 'notebooks')
        if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró qtrans/utils/qutils.py; ejecutá el cuadernillo dentro del proyecto')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Si el kernel ya tenía cargados estos módulos de una ejecución anterior, se
# recargan para que los cambios del código fuente se apliquen sin tener que
# reiniciar el kernel (si no, `import` reutiliza el módulo cacheado en
# ``sys.modules`` y falla con ImportError por atributos nuevos).
import importlib

import qtrans.utils.url_utils as _url_utils

importlib.reload(_url_utils)

import qtrans.utils.qutils as _qutils

importlib.reload(_qutils)

from qtrans.utils.qutils import quirk_circuit_info, quirk_to_qasm
from qtrans.utils.url_utils import encode_quirk_url, normalize_quirk_url, parse_quirk_url

def limpiar_url_quirk(url):
    """Normaliza una URL cruda de Quirk a un JSON estricto canónico.

    Quita las comillas externas, desescapa las comillas del JSON y recorta
    desde 'https://'. El JSON del circuito se re-serializa interpretando los
    escapes '\\uXXXX', de modo que 'Z^-\\u00bd' vuelve a ser 'Z^-½' y no
    'Z^-u00bd' (que Quirk no reconoce).

    Parámetros:
        - url (str): URL cruda (aunque traiga \\ y \" extra).

    Retorna:
        - str: la URL desde 'https://' con el JSON del circuito normalizado.
    """
    return normalize_quirk_url(url)


def formatear_url_quirk(url, percent_encoded=True):
    """Limpia la URL recibida y la devuelve en el formato pedido.

    Parámetros:
        - url (str): URL cruda (aunque traiga \\ y \" extra).
        - percent_encoded (bool): True devuelve la URL percent-encoded
          (apta como href); False devuelve la URL cruda (JSON directo).

    Retorna:
        - str: la URL en el formato indicado.
    """
    limpia = limpiar_url_quirk(url)
    if "circuit=" not in limpia:
        raise ValueError("La URL debe contener 'circuit=' con el JSON del circuito.")
    return encode_quirk_url(limpia) if percent_encoded else limpia


def validar_url_quirk(final_url):
    """Valida round-trip: decodifica la URL y reporta info del circuito."""
    try:
        circuito = parse_quirk_url(final_url)
    except Exception:
        print("❌ ERROR (no se pudo parsear el circuito)")
        return False
    if not (isinstance(circuito, dict) and "cols" in circuito):
        print("❌ ERROR (el circuito no tiene 'cols')")
        return False
    info = quirk_circuit_info(final_url)
    try:
        quirk_to_qasm(final_url)
    except Exception as exc:
        print(f"⚠️  AVISO: el circuito no se puede traducir ({exc})")
    print(f"✅ OK ({info['n_qubits']} qubits, {info['n_cols']} columnas)")
    return True


def mostrar_url_quirk(url, percent_encoded=True):
    """Limpia/formatea la URL recibida y muestra el enlace.

    Parámetros:
        - url (str): URL cruda de Quirk (aunque traiga \\ y \" extra).
        - percent_encoded (bool): True muestra/valida la URL percent-encoded
          (formato href de Quirk); False usa la URL cruda (JSON directo).

    Retorna:
        - str: la URL final en el formato indicado.
    """
    final_url = formatear_url_quirk(url, percent_encoded=percent_encoded)
    validar_url_quirk(final_url)
    href = encode_quirk_url(final_url) if percent_encoded else final_url
    display(HTML('<a href="' + href + '" target="_blank">ABRIR CIRCUITO EN QUIRK</a>'))
    return final_url


# --- Prueba: recibe directamente la URL cruda del circuito -------------------
mostrar_url_quirk(url_quirk)


✅ OK (2 qubits, 5 columnas)


'https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B%22H%22%2C1%5D%2C%5B%22Z%22%2C1%5D%2C%5B%22%5Cu2296%22%2C%22X%22%5D%2C%5B%22H%22%2C1%5D%2C%5B%22Measure%22%2C%22Measure%22%5D%5D%7D'

### Obtención de un json register a partir de una url para Quirk

Utilidad inversa de la anterior: a partir de una URL de Quirk (cruda o percent-encoded, con `circuit=<JSON>`) reconstruye el **json register crudo** tal y como figura en los objetos de `/input` (`{"desc": ..., "url": ...}`), con la URL en JSON estricto sin percent-encoding. Valida el round-trip reconstruyendo la URL y comparando la info del circuito.


In [3]:
# Pegamos la url tal y como figura en la url web de Quirk tool
url_quirk_percent_coded = (
    'https://algassert.com/quirk#circuit={%22cols%22:[[%22H%22,%22H%22,%22H%22,%22H%22,%22H%22,%22H%22],[1,1,%22Z^-%C2%BD%22,%22Z^-%C2%BD%22],[%22|/%E2%9F%A9%E2%9F%A8/|%22,%22|/%E2%9F%A9%E2%9F%A8/|%22,1,1,%22|/%E2%9F%A9%E2%9F%A8/|%22,%22|/%E2%9F%A9%E2%9F%A8/|%22],[%22Measure%22,%22Measure%22,%22Measure%22,%22Measure%22,%22Measure%22,%22Measure%22]]}'
)


In [ ]:
# ----------------------------------------------------------------------------
# Devuelve la info del circuito de la URL de la celda previa
# (``url_quirk_percent_coded``) y el objeto json del algoritmo.
# ----------------------------------------------------------------------------

import json
import sys
from pathlib import Path

# La raíz del proyecto se resuelve desde la ubicación del cuadernillo, igual
# que en QTrans.ipynb, para poder importar el paquete ``qtrans``.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / 'notebooks')
        if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró qtrans/utils/qutils.py; ejecutá el cuadernillo dentro del proyecto')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Si el kernel ya tenía cargados estos módulos de una ejecución anterior, se
# recargan para que los cambios del código fuente se apliquen sin tener que
# reiniciar el kernel (si no, `import` reutiliza el módulo cacheado en
# ``sys.modules`` y falla con ImportError por atributos nuevos).
import importlib

import qtrans.utils.url_utils as _url_utils

importlib.reload(_url_utils)

import qtrans.utils.qutils as _qutils

importlib.reload(_qutils)

from qtrans.utils.qutils import quirk_circuit_info
from qtrans.utils.url_utils import encode_quirk_url, normalize_quirk_url, parse_quirk_url


def info_circuito_quirk(url):
    """Devuelve la información del circuito codificado en una URL de Quirk.

    Parámetros:
        - url (str): URL de Quirk con `circuit=<JSON>` (cruda o percent-encoded).

    Retorna:
        - dict: info del circuito (`n_qubits`, `n_cols`, `gates`, ...).
    """
    limpia = normalize_quirk_url(url)
    if "circuit=" not in limpia:
        raise ValueError("La URL debe contener 'circuit=' con el JSON del circuito.")
    return quirk_circuit_info(limpia)


def url_quirk_a_json_register(url, desc="", percent_encoded=False):
    """Reconstruye el json register del algoritmo a partir de una URL de Quirk.

    Inversa de `encode_quirk_url`: limpia la url de entrada (aunque venga con
    comillas o backslashes de más), decodifica el `circuit=<JSON>` y lo
    re-serializa como JSON estricto (comillas dobles, ASCII escapado, sin
    espacios) para que la url del registro coincida con la de /input.

    Parámetros:
        - url (str): URL de Quirk con `circuit=<JSON>` (cruda o percent-encoded).
        - desc (str): descripción opcional para el registro.
        - percent_encoded (bool): True deja la `url` del registro
          percent-encoded; False la deja cruda (JSON directo, formato /input).

    Retorna:
        - dict: registro del algoritmo, `{"desc": ..., "url": ...}`.
    """
    limpia = normalize_quirk_url(url)
    if "circuit=" not in limpia:
        raise ValueError("La URL debe contener 'circuit=' con el JSON del circuito.")
    head, circuit_text = limpia.split("circuit=", 1)
    circuito = parse_quirk_url(limpia)  # valida que el JSON se decodifica
    circuito_json = json.dumps(circuito, ensure_ascii=True, separators=(",", ":"))
    url_salida = (
        encode_quirk_url(head + "circuit=" + circuito_json)
        if percent_encoded
        else head + "circuit=" + circuito_json
    )
    return {
        "desc": desc,
        "url": url_salida,
    }


# --- Resultado: info del circuito de la celda previa + objeto json ---------
info_circuito = info_circuito_quirk(url_quirk_percent_coded)
json_algoritmo = url_quirk_a_json_register(
    url_quirk_percent_coded,
    desc="Algoritmo de ejemplo ({} qubits, {} columnas)".format(
        info_circuito["n_qubits"], info_circuito["n_cols"]
    ),
    percent_encoded=True,
)
resultado = {"info": info_circuito, "json": json_algoritmo}
resultado
